In [29]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
EKSTRAKSI ULANG WAVEFORM VENEZUELA - NORMALISASI MCU-QUAKE
Mengikuti metode normalisasi dari paper Zhi Geng et al. (2025):
- Normalisasi per komponen dengan max absolut 9 detik setelah P
- Resample ke 100 Hz
- Detrend
- Output JSON 1C dan 3C
"""

import os
import sys
import json
import gc
import numpy as np
from pathlib import Path
from obspy import read, UTCDateTime
from obspy.signal.trigger import recursive_sta_lta
from tqdm import tqdm
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed

# =============================================
# KONFIGURASI
# =============================================
WAVEFORM_DIR = '/Volumes/Extreme SSD/venezuela_data_earthquake/waveform_venezuela_3c'
OUTPUT_JSON_1C = "/Volumes/Extreme SSD/venezuela_data_earthquake/json_venezuela/extracted_data_1c_venez.json"
OUTPUT_JSON_3C = "/Volumes/Extreme SSD/venezuela_data_earthquake/json_venezuela/extracted_data_3c_venez.json"

# Parameter MCU-Quake (dari paper)
SAMPLE_RATE = 100.0          # 100 Hz
SIG_DURATION = 7.0           # 7 detik sinyal setelah P
NOISE_DURATION = 7.0         # 7 detik noise sebelum P
NORM_WINDOW = 9.0            # 9 detik setelah P untuk normalisasi

# Parameter STA/LTA (sama seperti sebelumnya)
STA_WIN = 0.5
LTA_WIN = 8.0
TRIGGER_THRESHOLD = 2.5

# Parallel
MAX_WORKERS = 2
MIN_FILE_SIZE = 1024  # 1 KB

LOG_FILE = "extract_venezuela_mcu_norm.log"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler(LOG_FILE), logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

# =============================================
# FUNGSI BANTUAN
# =============================================

def pick_p_arrival(trace):
    """Deteksi P-wave arrival menggunakan STA/LTA (sama seperti sebelumnya)."""
    try:
        sr = trace.stats.sampling_rate
        sta_n = int(STA_WIN * sr)
        lta_n = int(LTA_WIN * sr)
        if len(trace.data) < lta_n + sta_n:
            return trace.stats.starttime + 5.0
        cft = recursive_sta_lta(trace.data, sta_n, lta_n)
        trigger = np.where(cft > TRIGGER_THRESHOLD)[0]
        if len(trigger) > 0:
            pick_idx = trigger[0]
            if pick_idx > int(2 * sr):
                return trace.stats.starttime + pick_idx / sr
        max_idx = np.argmax(np.abs(trace.data))
        if max_idx > 0:
            return trace.stats.starttime + max_idx / sr
    except:
        pass
    return trace.stats.starttime + 5.0

def get_preferred_trace(st, comp):
    """Pilih trace terbaik untuk komponen (Z, N, E) dengan prioritas BH > HH > EH > LH."""
    priority = ['BH', 'HH', 'EH', 'LH']
    for prefix in priority:
        ch = f"{prefix}{comp}"
        tr = st.select(channel=ch)
        if len(tr) > 0:
            return tr[0]
    # Fallback: cari channel yang berakhiran comp
    tr = st.select(channel=f"*{comp}")
    if len(tr) > 0:
        return tr[0]
    return None

def extract_component_mcu_norm(trace, p_time, comp_name):
    """
    Ekstrak sinyal dan noise untuk satu komponen dengan normalisasi MCU-Quake.
    - Potong sinyal 7 detik setelah P
    - Potong noise 7 detik sebelum P
    - Detrend
    - Resample ke 100 Hz
    - Normalisasi: bagi dengan max absolut dari jendela 9 detik setelah P (pada trace yang sudah di-resample)
    - Kembalikan (signal_list, noise_list)
    """
    try:
        # --- 1. Potong sinyal dan noise ---
        sig_start = p_time
        sig_end = p_time + SIG_DURATION
        noise_start = p_time - NOISE_DURATION
        noise_end = p_time

        tr_signal = trace.copy().trim(sig_start, sig_end)
        tr_noise = trace.copy().trim(noise_start, noise_end)

        # --- 2. Detrend ---
        tr_signal.detrend('simple')
        tr_noise.detrend('simple')

        # --- 3. Resample ke 100 Hz (sebelum normalisasi) ---
        if tr_signal.stats.sampling_rate != SAMPLE_RATE:
            tr_signal.resample(SAMPLE_RATE)
        if tr_noise.stats.sampling_rate != SAMPLE_RATE:
            tr_noise.resample(SAMPLE_RATE)

        # --- 4. Ambil jendela 9 detik setelah P untuk normalisasi ---
        norm_start = p_time
        norm_end = p_time + NORM_WINDOW
        tr_norm = trace.copy().trim(norm_start, norm_end)
        # Resample ke 100 Hz jika perlu
        if tr_norm.stats.sampling_rate != SAMPLE_RATE:
            tr_norm.resample(SAMPLE_RATE)

        # --- 5. Normalisasi per komponen dengan max absolut ---
        if len(tr_norm.data) > 0:
            max_val = np.max(np.abs(tr_norm.data))
        else:
            max_val = np.max(np.abs(tr_signal.data))
        if max_val < 1e-9:
            max_val = 1.0  # safety guard

        signal_data = tr_signal.data / max_val
        noise_data = tr_noise.data / max_val

        # --- 6. Potong/padding ke 700 sampel (7 detik * 100 Hz) ---
        target_len = int(SAMPLE_RATE * SIG_DURATION)
        def fix_length(data):
            if len(data) > target_len:
                return data[:target_len]
            elif len(data) < target_len:
                return np.pad(data, (0, target_len - len(data)), 'constant')
            return data

        return fix_length(signal_data).tolist(), fix_length(noise_data).tolist()

    except Exception as e:
        logger.debug(f"Error ekstraksi {comp_name}: {e}")
        return None, None

def process_file(file_path):
    """Proses satu file .mseed dengan normalisasi MCU-Quake."""
    try:
        st = read(str(file_path))
        if len(st) == 0:
            return None

        # Ambil trace Z, N, E
        trace_z = get_preferred_trace(st, 'Z')
        trace_n = get_preferred_trace(st, 'N')
        trace_e = get_preferred_trace(st, 'E')

        if trace_z is None:
            return None

        # P-wave picking dari Z
        p_time = pick_p_arrival(trace_z)

        # Ekstrak sinyal dan noise untuk Z (wajib)
        z_sig, z_noi = extract_component_mcu_norm(trace_z, p_time, 'Z')
        if z_sig is None:
            return None

        # Siapkan hasil
        result = {
            'event_id': file_path.stem,
            'network': trace_z.stats.network,
            'station': trace_z.stats.station,
            'p_arrival': str(p_time),
            'file': file_path.name,
            'has_z': True,
            'has_n': False,
            'has_e': False,
            'Z': z_sig,
            'Z_noise': z_noi,
        }

        # Ekstrak N jika ada
        if trace_n is not None:
            n_sig, n_noi = extract_component_mcu_norm(trace_n, p_time, 'N')
            if n_sig is not None:
                result['has_n'] = True
                result['N'] = n_sig
                result['N_noise'] = n_noi

        # Ekstrak E jika ada
        if trace_e is not None:
            e_sig, e_noi = extract_component_mcu_norm(trace_e, p_time, 'E')
            if e_sig is not None:
                result['has_e'] = True
                result['E'] = e_sig
                result['E_noise'] = e_noi

        # Bersihkan memory
        del st
        gc.collect()

        return result

    except Exception as e:
        logger.debug(f"Error processing {file_path.name}: {e}")
        return None

# =============================================
# MAIN
# =============================================
def main():
    logger.info("="*70)
    logger.info("🚀 EKSTRAKSI ULANG VENEZUELA - NORMALISASI MCU-QUAKE")
    logger.info("="*70)

    wave_dir = Path(WAVEFORM_DIR)
    all_files = list(wave_dir.glob("*.mseed"))
    all_files = [f for f in all_files if f.stat().st_size >= MIN_FILE_SIZE]
    logger.info(f"📁 Ditemukan {len(all_files)} file .mseed")

    # Load JSON existing (resume)
    data_1c = {}
    data_3c = {}
    if os.path.exists(OUTPUT_JSON_1C):
        with open(OUTPUT_JSON_1C, 'r') as f:
            data_1c = json.load(f)
        logger.info(f"📂 Load 1C existing: {len(data_1c)} entries")
    if os.path.exists(OUTPUT_JSON_3C):
        with open(OUTPUT_JSON_3C, 'r') as f:
            data_3c = json.load(f)
        logger.info(f"📂 Load 3C existing: {len(data_3c)} entries")

    # Filter file yang belum diproses
    files_to_process = []
    for f in all_files:
        key = f.stem
        if key not in data_1c:
            files_to_process.append(f)
    logger.info(f"📦 File baru: {len(files_to_process)}")

    if len(files_to_process) == 0:
        logger.info("✅ Semua file sudah diproses!")
        logger.info(f"📁 1C: {len(data_1c)}")
        logger.info(f"📁 3C: {len(data_3c)}")
        return

    success_1c = 0
    success_3c = 0
    failed = 0

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_file, f): f for f in files_to_process}
        with tqdm(total=len(futures), desc="Ekstraksi MCU-Norm") as pbar:
            for future in as_completed(futures):
                file_path = futures[future]
                result = future.result()

                if result:
                    key = result['event_id']
                    # Simpan ke 1C
                    data_1c[key] = {
                        'type': 'se',
                        'Z': result['Z'],
                        'Z_noise': result['Z_noise'],
                        'metadata': {
                            'network': result['network'],
                            'station': result['station'],
                            'p_arrival': result['p_arrival'],
                            'file': result['file']
                        }
                    }
                    success_1c += 1

                    # Jika memiliki N dan E, simpan ke 3C
                    if result['has_n'] and result['has_e']:
                        data_3c[key] = {
                            'type': 'se',
                            'Z': result['Z'],
                            'N': result['N'],
                            'E': result['E'],
                            'Z_noise': result['Z_noise'],
                            'N_noise': result['N_noise'],
                            'E_noise': result['E_noise'],
                            'metadata': {
                                'network': result['network'],
                                'station': result['station'],
                                'p_arrival': result['p_arrival'],
                                'file': result['file']
                            }
                        }
                        success_3c += 1
                else:
                    failed += 1

                pbar.update(1)

                # Checkpoint setiap 50 file
                if (success_1c + failed) % 50 == 0:
                    with open(OUTPUT_JSON_1C, 'w') as f:
                        json.dump(data_1c, f, indent=2)
                    with open(OUTPUT_JSON_3C, 'w') as f:
                        json.dump(data_3c, f, indent=2)
                    logger.info(f"💾 Checkpoint: 1C={len(data_1c)}, 3C={len(data_3c)}")

    # Simpan final
    with open(OUTPUT_JSON_1C, 'w') as f:
        json.dump(data_1c, f, indent=2)
    with open(OUTPUT_JSON_3C, 'w') as f:
        json.dump(data_3c, f, indent=2)

    logger.info("="*70)
    logger.info(f"✨ SELESAI!")
    logger.info(f"✅ 1C: {len(data_1c)} entries")
    logger.info(f"✅ 3C: {len(data_3c)} entries")
    logger.info(f"❌ Gagal: {failed}")
    logger.info(f"📂 Output 1C: {OUTPUT_JSON_1C}")
    logger.info(f"📂 Output 3C: {OUTPUT_JSON_3C}")
    logger.info("="*70)

if __name__ == "__main__":
    main()

2026-07-15 11:21:47,758 - INFO - ======================================================================
2026-07-15 11:21:47,759 - INFO - 🚀 EKSTRAKSI ULANG VENEZUELA - NORMALISASI MCU-QUAKE
2026-07-15 11:21:47,760 - INFO - ======================================================================
2026-07-15 11:21:47,816 - INFO - 📁 Ditemukan 2744 file .mseed
2026-07-15 11:21:48,220 - INFO - 📂 Load 1C existing: 1247 entries
2026-07-15 11:21:49,439 - INFO - 📂 Load 3C existing: 1247 entries
2026-07-15 11:21:49,440 - INFO - 📦 File baru: 1497


Ekstraksi MCU-Norm:   2%|▏         | 35/1497 [00:00<00:08, 167.50it/s]

2026-07-15 11:21:54,075 - INFO - 💾 Checkpoint: 1C=1247, 3C=1247


Ekstraksi MCU-Norm:   3%|▎         | 52/1497 [00:04<02:51,  8.45it/s] 

2026-07-15 11:21:58,362 - INFO - 💾 Checkpoint: 1C=1247, 3C=1247


...

2026-07-15 11:24:00,643 - INFO - 💾 Checkpoint: 1C=1324, 3C=1324


Ekstraksi MCU-Norm:  97%|█████████▋| 1450/1497 [02:12<00:03, 12.83it/s]

2026-07-15 11:24:07,650 - INFO - 💾 Checkpoint: 1C=1349, 3C=1349


Ekstraksi MCU-Norm: 100%|██████████| 1497/1497 [02:19<00:00, 10.73it/s]


2026-07-15 11:24:14,470 - INFO - ======================================================================
2026-07-15 11:24:14,471 - INFO - ✨ SELESAI!
2026-07-15 11:24:14,472 - INFO - ✅ 1C: 1372 entries
2026-07-15 11:24:14,472 - INFO - ✅ 3C: 1372 entries
2026-07-15 11:24:14,472 - INFO - ❌ Gagal: 1372
2026-07-15 11:24:14,472 - INFO - 📂 Output 1C: /Volumes/Extreme SSD/venezuela_data_earthquake/json_venezuela/extracted_data_1c_venez.json
2026-07-15 11:24:14,473 - INFO - 📂 Output 3C: /Volumes/Extreme SSD/venezuela_data_earthquake/json_venezuela/extracted_data_3c_venez.json
2026-07-15 11:24:14,473 - INFO - ======================================================================


In [28]:
import json

with open("/Volumes/Extreme SSD/venezuela_data_earthquake/json_venezuela/extracted_data_3c_venez.json", 'r') as f:
    data_3c = json.load(f)

print(f"Total entri 3C: {len(data_3c)}")

# Periksa 5 entri pertama
keys = list(data_3c.keys())[:5]
for key in keys:
    entry = data_3c[key]
    has_n = 'N' in entry and entry['N'] is not None
    has_e = 'E' in entry and entry['E'] is not None
    print(f"{key}: N={has_n}, E={has_e}")

Total entri 3C: 1247
GE_BOAB_20260707_084137: N=True, E=True
GE_BOAB_20260629_110103: N=True, E=True
G_FDFM_20260707_084137: N=True, E=True
G_FDFM_20260629_110103: N=True, E=True
GE_BOAB_20260626_221611: N=True, E=True


In [27]:
import json
with open("/Volumes/Extreme SSD/venezuela_data_earthquake/json_venezuela/extracted_data_3c_venez.json", 'r') as f:
    data = json.load(f)
print(f"Jumlah entri 3C: {len(data)}")
# Lihat satu contoh
key = list(data.keys())[0]
print(data[key].keys())

Jumlah entri 3C: 1247
dict_keys(['type', 'Z', 'N', 'E', 'Z_noise', 'N_noise', 'E_noise', 'metadata'])


In [ ]:
import json
import matplotlib.pyplot as plt

# Muat JSON 3C
with open("/Volumes/Extreme SSD/venezuela_data_earthquake/json_venezuela/extracted_data_3c.json", 'r') as f:
    data_3c = json.load(f)

print(f"Jumlah entri 3C: {len(data_3c)}")
key = list(data_3c.keys())[0]
entry = data_3c[key]
print(f"Contoh key: {key}")
print(f"Keys: {entry.keys()}")

# Plot contoh
plt.figure(figsize=(12, 4))
plt.plot(entry['Z'], label='Z', linewidth=1.5)
plt.plot(entry['N'], label='N', linewidth=1.5)
plt.plot(entry['E'], label='E', linewidth=1.5)
plt.legend()
plt.title(f"{key} - {entry['metadata']['station']}")
plt.xlabel('Sampel (0-700 = 0-7 detik)')
plt.ylabel('Amplitudo ternormalisasi')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
print(entry['metadata']['p_arrival'])

In [ ]:
import json

with open('/Volumes/Extreme SSD/venezuela_data_earthquake/json_venezuela/extracted_data_3c.json') as f:
    data = json.load(f)

print(f"Jumlah event: {len(data)}")
# Lihat satu contoh
key = list(data.keys())[0]
print(f"Contoh key: {key}")
print(f"Keys dalam entry: {data[key].keys()}")
print(f"Panjang sinyal Z: {len(data[key]['Z'])}")

In [ ]:
import json
import matplotlib.pyplot as plt

# Baca JSON 3C
with open('/Volumes/Extreme SSD/venezuela_data_earthquake/json_venezuela/extracted_data_3c.json', 'r') as f:
    data = json.load(f)

# Ambil contoh event
key = 'G_FDFM_20250925_035139'
entry = data[key]

# Buat plot 3 komponen
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
t = range(700)  # 7 detik pada 100 Hz

for i, (comp, color) in enumerate(zip(['Z', 'N', 'E'], ['r', 'g', 'b'])):
    axes[i].plot(t, entry[comp], color=color, label=f'Signal {comp}')
    axes[i].plot(t, entry[f'{comp}_noise'], color=color, linestyle='--', alpha=0.5, label=f'Noise {comp}')
    axes[i].legend(loc='upper right')
    axes[i].set_ylabel('Amplitudo (ternormalisasi)')

axes[-1].set_xlabel('Sampel (0-700 = 0-7 detik)')
plt.suptitle(f'Event: {key} | Stasiun: {entry["metadata"]["station"]}')
plt.tight_layout()
plt.show()

In [ ]:
import json
import matplotlib.pyplot as plt

# Baca JSON 3C
with open('/Volumes/Extreme SSD/venezuela_data_earthquake/json_venezuela/extracted_data_3c.json', 'r') as f:
    data = json.load(f)

# Ambil contoh event (ganti dengan key yang Anda inginkan)
key = 'G_FDFM_20250925_035139'
entry = data[key]

# Cek komponen mana yang tersedia
available_comps = [comp for comp in ['Z', 'N', 'E'] if entry.get(comp) is not None]
print(f"Komponen tersedia: {available_comps}")

if not available_comps:
    print("Tidak ada komponen yang tersedia untuk event ini.")
else:
    # Buat plot
    fig, axes = plt.subplots(len(available_comps), 1, figsize=(12, 6), sharex=True)
    if len(available_comps) == 1:
        axes = [axes]  # agar iterasi tetap berjalan

    t = range(700)  # 7 detik pada 100 Hz
    colors = {'Z': 'r', 'N': 'g', 'E': 'b'}

    for i, comp in enumerate(available_comps):
        signal = entry[comp]
        noise = entry[f'{comp}_noise']
        
        axes[i].plot(t, signal, color=colors[comp], label=f'Signal {comp}')
        axes[i].plot(t, noise, color=colors[comp], linestyle='--', alpha=0.5, label=f'Noise {comp}')
        axes[i].legend(loc='upper right')
        axes[i].set_ylabel('Amplitudo (ternormalisasi)')

    axes[-1].set_xlabel('Sampel (0-700 = 0-7 detik)')
    plt.suptitle(f'Event: {key} | Stasiun: {entry["metadata"]["station"]}')
    plt.tight_layout()
    plt.show()

In [ ]:
import json
from obspy import read

# Baca file .mseed langsung
file_path = "/Volumes/Extreme SSD/venezuela_data_earthquake/waveform_venezuela/CU_GRGR_20260627_192037.mseed"
st = read(file_path)
for tr in st:
    print(tr.stats.channel, tr.stats.sampling_rate, tr.stats.npts)

In [ ]:
st.detrend('demean')
st.plot()

In [ ]:
print(entry['metadata']['p_arrival'])

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
EKSTRAKSI WAVEFORM VENEZUELA - MENGGUNAKAN AR_PICK
Menghasilkan dua JSON: 1C (hanya Z) dan 3C (Z,N,E)
Metode picking: ar_pick (jika 3 komponen), fallback STA/LTA.
Normalisasi: max absolut 15 detik setelah P.
"""

import os
import sys
import json
import numpy as np
from pathlib import Path
from obspy import read, UTCDateTime
from obspy.signal.trigger import ar_pick, recursive_sta_lta
from tqdm import tqdm
import logging

# =============================================
# KONFIGURASI
# =============================================
WAVEFORM_DIR = "/Volumes/Extreme SSD/venezuela_data_earthquake/waveform_venezuela"
OUTPUT_DIR = "/Volumes/Extreme SSD/venezuela_data_earthquake/json_venezuela"
OUTPUT_JSON_1C = os.path.join(OUTPUT_DIR, "extracted_data_venezuela_1comp.json")
OUTPUT_JSON_3C = os.path.join(OUTPUT_DIR, "extracted_data_venezuela_3comp.json")

# Parameter preprocessing
SAMPLE_RATE = 100.0          # target sampling rate (Hz)
SIG_DURATION = 7.0           # durasi sinyal setelah P (detik)
NOISE_DURATION = 7.0         # durasi noise sebelum P (detik)
NORM_WINDOW = 15.0           # jendela normalisasi setelah P (detik) [diperbesar]

# Parameter untuk ar_pick
PICK_F1 = 1.0                # frekuensi rendah untuk filter picking (Hz)
PICK_F2 = 5.0                # frekuensi tinggi untuk filter picking (Hz)
LTA_P = 2.0                  # LTA window P (detik)
STA_P = 0.1                  # STA window P (detik)
LTA_S = 4.0                  # LTA window S (detik)
STA_S = 0.2                  # STA window S (detik)
M_P = 2                      # order AR untuk P
M_S = 2                      # order AR untuk S
L_P = 0.1                    # panjang window untuk P (detik)
L_S = 0.1                    # panjang window untuk S (detik)

# Fallback STA/LTA jika ar_pick gagal
STA_WIN = 0.5
LTA_WIN = 8.0
TRIGGER_THRESHOLD = 2.5

# Testing
MAX_FILES = None             # None untuk semua

# =============================================
# SETUP LOGGING
# =============================================
os.makedirs(OUTPUT_DIR, exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(os.path.join(OUTPUT_DIR, "extract_venezuela_arpick.log"))
    ]
)
logger = logging.getLogger(__name__)

# =============================================
# FUNGSI BANTUAN
# =============================================

def get_preferred_trace(st, comp):
    """
    Pilih trace terbaik untuk komponen tertentu (Z, N, E).
    Prioritas channel: HH > BH > EH > LH.
    Kembalikan trace atau None.
    """
    if comp not in ['Z', 'N', 'E']:
        return None
    # Daftar prioritas channel
    priority = ['HH', 'BH', 'EH', 'LH']
    for prefix in priority:
        channel = f"{prefix}{comp}"
        tr = st.select(channel=channel)
        if len(tr) > 0:
            return tr[0]
    # Coba cari dengan wildcard
    wildcard = f"*{comp}"
    tr = st.select(channel=wildcard)
    if len(tr) > 0:
        return tr[0]
    return None

def pick_p_arrival(trace_z, trace_n=None, trace_e=None):
    """
    Gunakan ar_pick jika tiga komponen tersedia, fallback ke STA/LTA pada Z.
    Kembalikan waktu P (UTCDateTime) dan metode yang digunakan.
    """
    try:
        if trace_n is not None and trace_e is not None:
            # Buat salinan untuk picking, filter bandpass
            z = trace_z.copy()
            n = trace_n.copy()
            e = trace_e.copy()
            z.filter('bandpass', freqmin=PICK_F1, freqmax=PICK_F2)
            n.filter('bandpass', freqmin=PICK_F1, freqmax=PICK_F2)
            e.filter('bandpass', freqmin=PICK_F1, freqmax=PICK_F2)
            
            p_pick, s_pick = ar_pick(
                z.data, n.data, e.data,
                samp_rate=z.stats.sampling_rate,
                f1=PICK_F1, f2=PICK_F2,
                lta_p=LTA_P, sta_p=STA_P,
                lta_s=LTA_S, sta_s=STA_S,
                m_p=M_P, m_s=M_S,
                l_p=L_P, l_s=L_S,
                s_pick=True
            )
            if p_pick is not None:
                p_time = z.stats.starttime + p_pick
                return p_time, 'ar_pick'
    except Exception as e:
        logger.debug(f"ar_pick failed: {e}")
    
    # Fallback: STA/LTA pada Z
    try:
        tr = trace_z.copy()
        sr = tr.stats.sampling_rate
        sta_n = int(STA_WIN * sr)
        lta_n = int(LTA_WIN * sr)
        if len(tr.data) > lta_n + sta_n:
            cft = recursive_sta_lta(tr.data, sta_n, lta_n)
            trigger = np.where(cft > TRIGGER_THRESHOLD)[0]
            if len(trigger) > 0:
                pick_idx = trigger[0]
                if pick_idx > int(2 * sr):
                    p_time = tr.stats.starttime + pick_idx / sr
                    return p_time, 'sta_lta'
    except Exception as e:
        logger.debug(f"STA/LTA fallback failed: {e}")
    
    # Ultimate fallback: 5 detik setelah start
    p_time = trace_z.stats.starttime + 5.0
    return p_time, 'fallback'

def extract_component(trace, p_time):
    """Ekstrak sinyal dan noise dari satu trace komponen."""
    try:
        # Sinyal: 7 detik setelah P
        sig_start = p_time
        sig_end = p_time + SIG_DURATION
        tr_sig = trace.copy().trim(sig_start, sig_end)
        
        # Noise: 7 detik sebelum P
        noise_start = p_time - NOISE_DURATION
        noise_end = p_time
        tr_noise = trace.copy().trim(noise_start, noise_end)
        
        # Detrend
        tr_sig.detrend('simple')
        tr_noise.detrend('simple')
        
        # Resample ke target rate
        if tr_sig.stats.sampling_rate != SAMPLE_RATE:
            tr_sig.resample(SAMPLE_RATE)
        if tr_noise.stats.sampling_rate != SAMPLE_RATE:
            tr_noise.resample(SAMPLE_RATE)
        
        # Normalisasi: max absolut dalam 15 detik setelah P
        tr_norm = trace.copy().trim(p_time, p_time + NORM_WINDOW)
        if len(tr_norm.data) == 0:
            tr_norm = tr_sig.copy()
        max_val = np.max(np.abs(tr_norm.data))
        if max_val == 0:
            max_val = 1.0
        
        sig_data = tr_sig.data / max_val
        noise_data = tr_noise.data / max_val
        
        # Potong/padding ke panjang tetap (700 sampel)
        target_len = int(SAMPLE_RATE * SIG_DURATION)
        def fix_length(data):
            if len(data) > target_len:
                return data[:target_len]
            elif len(data) < target_len:
                return np.pad(data, (0, target_len - len(data)), 'constant')
            return data
        
        return fix_length(sig_data).tolist(), fix_length(noise_data).tolist()
    except Exception as e:
        logger.debug(f"Extraction error for {trace.stats.channel}: {e}")
        return None, None

def process_file(file_path):
    """
    Proses satu file .mseed.
    Return: (data_1c, data_3c, metadata) atau (None, None, None)
    """
    try:
        st = read(str(file_path))
        if len(st) == 0:
            return None, None, None
        
        # Pilih trace terbaik untuk Z, N, E
        trace_z = get_preferred_trace(st, 'Z')
        trace_n = get_preferred_trace(st, 'N')
        trace_e = get_preferred_trace(st, 'E')
        
        if trace_z is None:
            logger.warning(f"{file_path.name}: Tidak ada komponen Z, dilewati.")
            return None, None, None
        
        # Picking
        p_time, pick_method = pick_p_arrival(trace_z, trace_n, trace_e)
        
        # Ekstrak untuk setiap komponen yang ada
        comps = {}
        comps_noise = {}
        for comp, tr in [('Z', trace_z), ('N', trace_n), ('E', trace_e)]:
            if tr is not None:
                sig, noise = extract_component(tr, p_time)
                if sig is not None and noise is not None:
                    comps[comp] = sig
                    comps_noise[comp] = noise
        
        # Minimal harus ada Z
        if 'Z' not in comps:
            logger.warning(f"{file_path.name}: Ekstraksi Z gagal, dilewati.")
            return None, None, None
        
        # Metadata
        metadata = {
            'network': trace_z.stats.network,
            'station': trace_z.stats.station,
            'channel_z': trace_z.stats.channel,
            'p_arrival': str(p_time),
            'pick_method': pick_method,
            'file': file_path.name
        }
        if trace_n:
            metadata['channel_n'] = trace_n.stats.channel
        if trace_e:
            metadata['channel_e'] = trace_e.stats.channel
        
        # Data 1C (hanya Z)
        data_1c = {
            'type': 'se',
            'Z': comps['Z'],
            'Z_noise': comps_noise['Z'],
            'metadata': metadata
        }
        
        # Data 3C
        data_3c = {
            'type': 'se',
            'Z': comps.get('Z'),
            'N': comps.get('N'),
            'E': comps.get('E'),
            'Z_noise': comps_noise.get('Z'),
            'N_noise': comps_noise.get('N'),
            'E_noise': comps_noise.get('E'),
            'metadata': metadata
        }
        
        return data_1c, data_3c, metadata
    except Exception as e:
        logger.error(f"Error processing {file_path.name}: {e}")
        return None, None, None

def main():
    logger.info("="*60)
    logger.info("🚀 EKSTRAKSI WAVEFORM VENEZUELA - AR_PICK")
    logger.info("="*60)
    
    wave_dir = Path(WAVEFORM_DIR)
    all_files = list(wave_dir.glob("*.mseed"))
    # Filter file metadata macOS
    all_files = [f for f in all_files if not f.name.startswith('._')]
    logger.info(f"📁 Ditemukan {len(all_files)} file .mseed valid")
    
    if MAX_FILES and len(all_files) > MAX_FILES:
        all_files = all_files[:MAX_FILES]
        logger.info(f"⚠️ Hanya memproses {MAX_FILES} file pertama (testing).")
    
    # Load existing JSON (resume)
    data_1c = {}
    data_3c = {}
    if os.path.exists(OUTPUT_JSON_1C):
        with open(OUTPUT_JSON_1C, 'r') as f:
            data_1c = json.load(f)
        logger.info(f"📂 Load 1C existing: {len(data_1c)} entries")
    if os.path.exists(OUTPUT_JSON_3C):
        with open(OUTPUT_JSON_3C, 'r') as f:
            data_3c = json.load(f)
        logger.info(f"📂 Load 3C existing: {len(data_3c)} entries")
    
    success = 0
    failed = 0
    skipped = 0
    
    for file_path in tqdm(all_files, desc="Memproses"):
        key = file_path.stem  # misal "CU_GRGR_20260628_084610"
        
        # Jika sudah ada di kedua JSON, skip
        if key in data_1c and key in data_3c:
            skipped += 1
            continue
        
        result_1c, result_3c, metadata = process_file(file_path)
        if result_1c is not None and result_3c is not None:
            data_1c[key] = result_1c
            data_3c[key] = result_3c
            success += 1
        else:
            failed += 1
        
        # Simpan checkpoint setiap 50 file
        if (success + failed) % 50 == 0:
            with open(OUTPUT_JSON_1C, 'w') as f:
                json.dump(data_1c, f, indent=2)
            with open(OUTPUT_JSON_3C, 'w') as f:
                json.dump(data_3c, f, indent=2)
            logger.info(f"💾 Checkpoint: {success} berhasil, {failed} gagal")
    
    # Simpan final
    with open(OUTPUT_JSON_1C, 'w') as f:
        json.dump(data_1c, f, indent=2)
    with open(OUTPUT_JSON_3C, 'w') as f:
        json.dump(data_3c, f, indent=2)
    
    logger.info("="*60)
    logger.info(f"✨ SELESAI! Berhasil: {success}, Gagal: {failed}, Skipped: {skipped}")
    logger.info(f"📁 1C JSON: {len(data_1c)} entries -> {OUTPUT_JSON_1C}")
    logger.info(f"📁 3C JSON: {len(data_3c)} entries -> {OUTPUT_JSON_3C}")
    logger.info("="*60)

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
VALIDASI JSON HASIL EKSTRAKSI WAVEFORM VENEZUELA
Memeriksa:
1. Kelengkapan kunci dan metadata.
2. Panjang sinyal dan noise (harus 700 sampel).
3. Keberadaan nilai None pada komponen kritis (Z).
4. Sinyal datar (std == 0) atau terlalu kecil.
5. Statistik amplitudo (mean, std, min, max).
6. Ringkasan per stasiun.
"""

import os
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
import logging

# =============================================
# KONFIGURASI
# =============================================
JSON_1C_PATH = "/Volumes/Extreme SSD/venezuela_data_earthquake/json_venezuela/extracted_data_venezuela_1comp.json"
JSON_3C_PATH = "/Volumes/Extreme SSD/venezuela_data_earthquake/json_venezuela/extracted_data_venezuela_3comp.json"
OUTPUT_CSV = "/Volumes/Extreme SSD/venezuela_data_earthquake/validation_json_report.csv"
OUTPUT_SUMMARY = "/Volumes/Extreme SSD/venezuela_data_earthquake/validation_json_summary.txt"
PLOT_OUTPUT = "/Volumes/Extreme SSD/venezuela_data_earthquake/validation_plots"  # opsional

# =============================================
# SETUP LOGGING
# =============================================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

# =============================================
# FUNGSI VALIDASI
# =============================================

def check_entry(entry, comps_expected=['Z','N','E']):
    """
    Validasi satu entry.
    Return dict dengan status dan metrik.
    """
    result = {
        'valid': True,
        'missing_keys': [],
        'length_ok': True,
        'z_present': False,
        'z_all_none': False,
        'z_std': None,
        'n_present': False,
        'e_present': False,
        'z_mean': None,
        'z_max': None,
        'z_min': None,
        'noise_mean': None,
        'noise_std': None,
        'all_components': []
    }

    # Periksa kunci yang diharapkan
    required_keys = ['type', 'metadata']
    if 'Z' in entry:
        required_keys.append('Z')
    if 'Z_noise' in entry:
        required_keys.append('Z_noise')
    for k in required_keys:
        if k not in entry:
            result['missing_keys'].append(k)
            result['valid'] = False

    # Periksa metadata
    if 'metadata' in entry:
        meta = entry['metadata']
        for mkey in ['network','station','p_arrival','file']:
            if mkey not in meta:
                result['missing_keys'].append(f'metadata.{mkey}')
                result['valid'] = False

    # Cek komponen Z (harus ada dan tidak None)
    z_data = entry.get('Z')
    z_noise = entry.get('Z_noise')
    if z_data is None or z_noise is None:
        result['z_present'] = False
        result['valid'] = False
    else:
        result['z_present'] = True
        # Cek panjang
        if len(z_data) != 700 or len(z_noise) != 700:
            result['length_ok'] = False
            result['valid'] = False
        # Hitung statistik Z
        z_arr = np.array(z_data)
        z_noise_arr = np.array(z_noise)
        result['z_std'] = float(z_arr.std())
        result['z_mean'] = float(z_arr.mean())
        result['z_max'] = float(z_arr.max())
        result['z_min'] = float(z_arr.min())
        result['noise_mean'] = float(z_noise_arr.mean())
        result['noise_std'] = float(z_noise_arr.std())
        # Deteksi sinyal datar
        if result['z_std'] == 0.0:
            result['valid'] = False

    # Cek komponen N dan E (jika ada)
    for comp in ['N','E']:
        data = entry.get(comp)
        if data is not None and len(data) == 700:
            if comp == 'N':
                result['n_present'] = True
            else:
                result['e_present'] = True
            # Cek apakah semua None? (sudah teratasi karena data not None)
        else:
            # Jika data None atau panjang tidak 700, tidak dianggap error tapi dicatat
            pass

    result['all_components'] = [c for c in ['Z','N','E'] if entry.get(c) is not None and len(entry.get(c)) == 700]

    return result

def validate_json(json_path, expected_comps):
    """
    Validasi file JSON, kembalikan list hasil per entri.
    """
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    logger.info(f"📂 Memproses {json_path} dengan {len(data)} entri")
    results = []
    for key, entry in data.items():
        res = check_entry(entry, expected_comps)
        res['entry_key'] = key
        res['network'] = entry.get('metadata', {}).get('network', 'unknown')
        res['station'] = entry.get('metadata', {}).get('station', 'unknown')
        res['p_arrival'] = entry.get('metadata', {}).get('p_arrival', '')
        res['file'] = entry.get('metadata', {}).get('file', '')
        results.append(res)
    return results, data

def generate_summary(results, label):
    """Buat ringkasan statistik dari hasil validasi."""
    total = len(results)
    valid = sum(1 for r in results if r['valid'])
    z_present = sum(1 for r in results if r['z_present'])
    z_std_vals = [r['z_std'] for r in results if r['z_std'] is not None]
    has_n = sum(1 for r in results if r['n_present'])
    has_e = sum(1 for r in results if r['e_present'])
    all_3comp = sum(1 for r in results if len(r['all_components']) == 3)
    
    summary = {
        'label': label,
        'total': total,
        'valid': valid,
        'invalid': total - valid,
        'z_present': z_present,
        'z_missing': total - z_present,
        'z_std_mean': float(np.mean(z_std_vals)) if z_std_vals else None,
        'z_std_median': float(np.median(z_std_vals)) if z_std_vals else None,
        'has_n': has_n,
        'has_e': has_e,
        '3comp': all_3comp,
        '2comp': total - all_3comp - (total - z_present),  # estimasi
        '1comp': total - all_3comp - has_n - has_e
    }
    return summary

def main():
    logger.info("="*60)
    logger.info("🔍 VALIDASI JSON HASIL EKSTRAKSI")
    logger.info("="*60)

    # Validasi JSON 1C
    res_1c, data_1c = validate_json(JSON_1C_PATH, ['Z'])
    summary_1c = generate_summary(res_1c, "1C")
    logger.info(f"📊 1C: {summary_1c['valid']} valid dari {summary_1c['total']}")

    # Validasi JSON 3C
    res_3c, data_3c = validate_json(JSON_3C_PATH, ['Z','N','E'])
    summary_3c = generate_summary(res_3c, "3C")
    logger.info(f"📊 3C: {summary_3c['valid']} valid dari {summary_3c['total']}")

    # Gabungkan hasil untuk CSV
    df_1c = pd.DataFrame(res_1c)
    df_3c = pd.DataFrame(res_3c)
    df_1c['json_type'] = '1C'
    df_3c['json_type'] = '3C'
    df = pd.concat([df_1c, df_3c], ignore_index=True)

    # Simpan laporan detail CSV
    df.to_csv(OUTPUT_CSV, index=False)
    logger.info(f"💾 Laporan detail disimpan di: {OUTPUT_CSV}")

    # Buat summary teks
    with open(OUTPUT_SUMMARY, 'w') as f:
        f.write("="*60 + "\n")
        f.write("VALIDASI JSON EKSTRAKSI WAVEFORM VENEZUELA\n")
        f.write("="*60 + "\n\n")

        for summary, label in [(summary_1c, "1C"), (summary_3c, "3C")]:
            f.write(f"--- {label} ---\n")
            f.write(f"Total entries: {summary['total']}\n")
            f.write(f"Valid: {summary['valid']}\n")
            f.write(f"Invalid: {summary['invalid']}\n")
            f.write(f"Komponen Z tersedia: {summary['z_present']}\n")
            f.write(f"Komponen Z hilang: {summary['z_missing']}\n")
            f.write(f"Memiliki komponen N: {summary['has_n']}\n")
            f.write(f"Memiliki komponen E: {summary['has_e']}\n")
            f.write(f"Memiliki 3 komponen: {summary['3comp']}\n")
            if summary['z_std_mean'] is not None:
                f.write(f"Rata-rata std sinyal Z: {summary['z_std_mean']:.6f}\n")
                f.write(f"Median std sinyal Z: {summary['z_std_median']:.6f}\n")
            f.write("\n")

        # Daftar entri yang tidak valid (jika ada)
        invalid_entries = df[df['valid'] == False]
        if len(invalid_entries) > 0:
            f.write("\n--- ENTRI TIDAK VALID ---\n")
            for idx, row in invalid_entries.iterrows():
                f.write(f"  {row['entry_key']} ({row['json_type']}): {row.get('missing_keys', [])}\n")

        f.write("\n" + "="*60 + "\n")

    logger.info(f"💾 Ringkasan disimpan di: {OUTPUT_SUMMARY}")

    # Tampilkan beberapa statistik
    logger.info("\n📊 RINGKASAN:")
    logger.info(f"  1C: {summary_1c['valid']} valid / {summary_1c['total']}")
    logger.info(f"  3C: {summary_3c['valid']} valid / {summary_3c['total']}")
    logger.info(f"  Entri 3 komponen: {summary_3c['3comp']}")
    logger.info(f"  Rata-rata std Z (3C): {summary_3c['z_std_mean']:.6f}")

    logger.info("✅ VALIDASI SELESAI")

if __name__ == "__main__":
    main()

In [ ]:
import json

with open("/Volumes/Extreme SSD/venezuela_data_earthquake/json_venezuela/extracted_data_venezuela_3comp.json", 'r') as f:
    data = json.load(f)

key = list(data.keys())[0]  # ambil pertama
entry = data[key]
print("Keys:", entry.keys())
print("Z type:", type(entry['Z']), "len:", len(entry['Z']) if entry['Z'] else None)
print("N type:", type(entry['N']), "len:", len(entry['N']) if entry['N'] else None)
print("E type:", type(entry['E']), "len:", len(entry['E']) if entry['E'] else None)
print("Z_noise len:", len(entry['Z_noise']) if entry['Z_noise'] else None)

In [ ]:
from obspy import read

file_path = "/Volumes/Extreme SSD/venezuela_data_earthquake/waveform_venezuela/CU_GRGR_20260627_192037.mseed"
st = read(file_path)
for tr in st:
    print(tr.stats.channel, tr.stats.sampling_rate)